# 03 — Feature inspection and statistical discrimination

**Goal.** Decide, before any modelling, whether the LLM features are worth modelling with.

We ask four questions of each feature:

1. **Does it vary?** A feature that assigns the same value to almost everyone carries no information.
2. **Is it just transcript length?** Notebook 01 showed the positive group says less. Any
   feature that tracks length will look predictive without measuring a linguistic construct.
3. **Is it associated with the label?** Chi-squared test, corrected for testing ten features at once.
4. **Is it redundant?** Two features measuring the same thing add noise, not signal.

Everything here uses `train/` only.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency, false_discovery_control
from sklearn.metrics import roc_auc_score

pd.set_option("display.width", 200)

corpus = pd.read_csv("outputs/corpus.csv")
features = pd.read_csv("outputs/features_v1_train.csv")
FEATURE_COLS = [c for c in features.columns if c not in ("File", "Class", "raw_llm_output")]

# join the feature table onto the transcripts so we can compare features against length
train = corpus[corpus.split == "train"].merge(
    features[["File", "Class"] + FEATURE_COLS], left_on="file", right_on="File", validate="1:1"
)

# consistency check: the folder label and the CSV Class column must agree
assert ((train.Class == "positive").astype(int) == train.label).all(), "label mismatch!"
y = train.label.astype(int).values
print(f"{len(train)} training documents, {y.sum()} positive — labels agree between sources")

241 training documents, 70 positive — labels agree between sources


## Question 1 — does each feature actually vary?

`top_share` is the fraction of documents receiving the single most common value. Near 1.0
means the model put almost everyone in one box. Normalised entropy near 0 means the same
thing on a 0–1 scale.

In [2]:
def level_usage(df, col):
    counts = df[col].value_counts()
    p = counts / counts.sum()
    entropy = float(-(p * np.log(p)).sum() / np.log(max(len(counts), 2)))
    return len(counts), counts.index[0], counts.iloc[0] / len(df), entropy

usage = pd.DataFrame(
    [(c, *level_usage(train, c)) for c in FEATURE_COLS],
    columns=["feature", "n_levels", "top_value", "top_share", "entropy"],
).sort_values("top_share", ascending=False)
print(usage.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

                           feature  n_levels        top_value  top_share  entropy
      spatial_organization_pattern         4      random_jump      0.921    0.261
emotional_interpretation_inference         4  purely_physical      0.622    0.693
          entity_specificity_level         3 basic_attributes      0.614    0.845
     action_verb_tense_consistency         4   strict_present      0.581    0.617
               narrative_coherence         4  fragmented_list      0.519    0.831
          quantification_precision         4   estimates_only      0.477    0.886
  lexical_variation_and_repetition         4    low_variation      0.432    0.793
            discourse_marker_usage         5           sparse      0.423    0.871
               uncertainty_markers         4       occasional      0.369    0.923
   self_referential_metacommentary         5           absent      0.353    0.848


## Question 2 — is the feature a proxy for transcript length?

We compute η² (eta squared): the proportion of variance in word count explained by the
feature's levels. A high value means the feature is largely restating how much the person
said. We use 0.30 as a rough alarm threshold.

In [3]:
def eta_squared(df, col, target="n_word"):
    grand = df[target].mean()
    grouped = df.groupby(col)[target]
    between = ((grouped.mean() - grand) ** 2 * grouped.size()).sum()
    total = ((df[target] - grand) ** 2).sum()
    return float(between / total)

eta = pd.DataFrame(
    [(c, eta_squared(train, c)) for c in FEATURE_COLS], columns=["feature", "eta2_length"]
).sort_values("eta2_length", ascending=False)
print(eta.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

                           feature  eta2_length
          entity_specificity_level        0.350
emotional_interpretation_inference        0.204
          quantification_precision        0.187
               narrative_coherence        0.151
      spatial_organization_pattern        0.109
               uncertainty_markers        0.108
            discourse_marker_usage        0.081
  lexical_variation_and_repetition        0.038
   self_referential_metacommentary        0.037
     action_verb_tense_consistency        0.033


## Question 3 — is the feature associated with the diagnosis?

A chi-squared test of independence per feature. Because we test ten features at once, we apply
a Benjamini–Hochberg correction: `q` is the false-discovery-rate-adjusted p-value, and `q < 0.05`
is the honest threshold for calling a feature associated.

We also report the best AUC achievable from any single level of the feature, which says how
much predictive power one feature has on its own.

In [4]:
def association(df, col, y):
    try:
        p = chi2_contingency(pd.crosstab(df[col], y))[1]
    except ValueError:
        p = np.nan
    aucs = []
    for level in df[col].dropna().unique():
        ind = (df[col] == level).astype(int).values
        if 0 < ind.sum() < len(ind):
            a = roc_auc_score(y, ind)
            aucs.append(max(a, 1 - a))       # direction does not matter
    return p, (max(aucs) if aucs else np.nan)

assoc = pd.DataFrame(
    [(c, *association(train, c, y)) for c in FEATURE_COLS],
    columns=["feature", "chi2_p", "best_level_auc"],
)
assoc["q_bh"] = false_discovery_control(assoc.chi2_p.fillna(1).values, method="bh")
assoc = assoc.sort_values("q_bh")
print(assoc.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
print(f"\nfeatures with q < 0.05: {(assoc.q_bh < 0.05).sum()} of {len(assoc)}")

                           feature  chi2_p  best_level_auc   q_bh
   self_referential_metacommentary  0.0001          0.6087 0.0012
     action_verb_tense_consistency  0.0003          0.6376 0.0013
            discourse_marker_usage  0.0013          0.5840 0.0042
               narrative_coherence  0.0093          0.5875 0.0234
  lexical_variation_and_repetition  0.0179          0.5990 0.0358
emotional_interpretation_inference  0.0232          0.5949 0.0386
               uncertainty_markers  0.0312          0.5957 0.0446
          entity_specificity_level  0.0952          0.5507 0.1190
      spatial_organization_pattern  0.3613          0.5254 0.4015
          quantification_precision  0.6522          0.5343 0.6522

features with q < 0.05: 7 of 10


## Putting it together — one verdict per feature

In [5]:
report = usage.merge(eta, on="feature").merge(assoc, on="feature").sort_values("q_bh")

def verdict(r):
    if r.top_share > 0.85 or r.entropy < 0.40:
        return "DEGENERATE — no variation"
    if r.eta2_length > 0.30:
        return "LENGTH PROXY"
    if r.q_bh < 0.05:
        return "keep"
    return "no association"

report["verdict"] = report.apply(verdict, axis=1)
cols = ["feature", "n_levels", "top_share", "entropy", "eta2_length", "q_bh", "best_level_auc", "verdict"]
print(report[cols].to_string(index=False, float_format=lambda v: f"{v:.3f}"))
report.to_csv("outputs/feature_report_v1.csv", index=False)

                           feature  n_levels  top_share  entropy  eta2_length  q_bh  best_level_auc                   verdict
   self_referential_metacommentary         5      0.353    0.848        0.037 0.001           0.609                      keep
     action_verb_tense_consistency         4      0.581    0.617        0.033 0.001           0.638                      keep
            discourse_marker_usage         5      0.423    0.871        0.081 0.004           0.584                      keep
               narrative_coherence         4      0.519    0.831        0.151 0.023           0.588                      keep
  lexical_variation_and_repetition         4      0.432    0.793        0.038 0.036           0.599                      keep
emotional_interpretation_inference         4      0.622    0.693        0.204 0.039           0.595                      keep
               uncertainty_markers         4      0.369    0.923        0.108 0.045           0.596                   

## Question 4 — are the features redundant?

Cramér's V between every pair of features: 0 = unrelated, 1 = identical. Pairs above ~0.5 are
measuring much the same thing.

In [6]:
def cramers_v(a, b):
    ct = pd.crosstab(a, b)
    chi2 = chi2_contingency(ct)[0]
    n = ct.values.sum()
    r, k = ct.shape
    return float(np.sqrt((chi2 / n) / max(min(r - 1, k - 1), 1)))

V = pd.DataFrame(
    [[cramers_v(train[a], train[b]) if a != b else 1.0 for b in FEATURE_COLS] for a in FEATURE_COLS],
    index=FEATURE_COLS, columns=FEATURE_COLS,
)
pairs = [
    (a, b, V.loc[a, b])
    for i, a in enumerate(FEATURE_COLS) for b in FEATURE_COLS[i + 1:]
]
top = sorted(pairs, key=lambda t: -t[2])[:6]
print("most strongly associated feature pairs (Cramér's V):")
for a, b, v in top:
    print(f"  {v:.3f}   {a}  ~  {b}")

most strongly associated feature pairs (Cramér's V):
  0.523   uncertainty_markers  ~  discourse_marker_usage
  0.471   self_referential_metacommentary  ~  discourse_marker_usage
  0.433   narrative_coherence  ~  entity_specificity_level
  0.409   narrative_coherence  ~  lexical_variation_and_repetition
  0.409   uncertainty_markers  ~  self_referential_metacommentary
  0.369   entity_specificity_level  ~  lexical_variation_and_repetition


## How much information is in ten categorical variables at all?

If nearly every document has a unique combination of feature values, a flexible model can
memorise the training set instead of generalising. This is a direct warning about tree
ensembles at this sample size.

In [7]:
combos = train[FEATURE_COLS].drop_duplicates().shape[0]
grp = train.groupby(FEATURE_COLS).label.agg(["size", "nunique"])
conflict = grp[(grp["size"] > 1) & (grp["nunique"] > 1)]
print(f"distinct feature vectors : {combos} out of {len(train)} documents")
print(f"vectors seen more than once with BOTH labels: {conflict['size'].sum()} documents")

distinct feature vectors : 221 out of 241 documents
vectors seen more than once with BOTH labels: 15 documents


### Summary — what to say about this

Read the verdict column. The pattern to look for, and what it means:

* **Several features pass `q < 0.05`.** The *constructs* the LLM proposed are picking up
  something real. The problem, if there is one, is not the ideas.
* **But the best single-level AUC is low** (around 0.6). Each feature separates the groups only
  weakly, because a three-to-five level categorical label is a very coarse summary of a
  60-word transcript.
* **A degenerate feature is a prompt bug, not a finding.** If the model assigns one value to
  90% of documents, the level definitions were not usable and should be redesigned or dropped.
* **A length proxy is a trap.** It will contribute to model performance while telling you
  nothing a word count would not.
* **Near-unique feature vectors** mean gradient boosting will overfit. Notebook 04 shows this
  happening.

This is what motivates the v2 feature design in notebook 06: keep the constructs, replace the
coarse categorical encoding with counts and evidence.